# NBL-survival
For response to reviewers.
Is there any difference in survival between extra- and intrachromosomally amplified neuroblastomas?

## Dependencies
Dependencies:  
r-survival.yml  
Run `preprocess-rodriguezfos-data.ipynb`

## Results
ecDNA has (barely) significantly worse outcomes than intrachromosomal amplification, both in the
Kaplan-Meier model (adjusted p = 0.042) and in the Cox model controlling for sex, age and amplification ONLY (p = 0.048). Including MYCN amplification as a covariate, using either the Rodriguez annotations or the AmpliconClassifier, renders the effect of ecDNA nonsignificant. Figures saved to `./out`.

## TODO
- Association test comparing incidence of ecDNA vs chromosomal amp for MYCN vs other loci?
- Follow up with Elias and Anton about discrepant ecDNA classifications.

In [ ]:
Sys.setenv(LANGUAGE = "en") # set language to "ja" if you prefer

library(tidyverse)
library(readxl)
library(dplyr)
library(stringr)
library(naniar) #for replace with Nas function
library(survival)
library(survminer)
library(RColorBrewer)
library(janitor)
library(gt)
library(gtsummary)
library(ggsurvfit)
library(extrafont)
library(svglite)

# imports from external file
imports <- new.env()
source("survival-data-imports.R", local = imports)
plotting <- new.env()
source("survival-plots.R",local=plotting)
source("../../src/plotting.R",local=plotting)

sessionInfo()

In [ ]:
extrafont::font_import(pattern="Arial",prompt=FALSE)
extrafont::loadfonts()

In [ ]:
dir.create('nbl', showWarnings = FALSE)

nbl_survival_data = 'nbl/processed_nbl_survival_data.tsv'
data <- imports$load_nbl_data(nbl_survival_data)

In [ ]:
data %>%head()

### KM of 175 NBL, stratified by amplicon class

KM pairwise log-rank tests:
```
            no amplification chromosomal
chromosomal 0.216            -          
ecDNA       1.7e-07          0.042 
```

In [ ]:
formula = Surv(OS_months_5y, OS_status_5y) ~ amplicon_class
km = survfit2(formula=formula, data = data )
plotting$km_plot(km)
plotting$save_ggplot("km_nbl_5year")
logrank <- pairwise_survdiff(formula,data,p.adjust.method="BH",rho=0)
logrank

### Fully parameterized Cox model, n=156
Hazard ratios:
```
ecDNA: 2.79; p=0.048
amplification: 2.47; p=0.10
```
`cox.zph` proportionality tests:
```
                 chisq df     p
ecDNA_status      1.69  1 0.194
amp_status        2.97  1 0.085
sex               0.94  1 0.332
age_at_diagnosis  1.44  1 0.230
GLOBAL            3.95  4 0.413
```

In [ ]:
# Cox model
m4 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + sex + age_at_diagnosis, data = data)
#plotting$cox_plot(m4,data)
#plotting$save_ggplot("cox_forest_nbl",width=6,height=3)
p <- plotting$forest_coxph(m4)
m4

plotting$show_forestploter(p)
plotting$save_forestploter(p,'out/cox_forest_nbl.png')
plotting$save_forestploter(p,'out/cox_forest_nbl.svg')

In [ ]:
zph = cox.zph(m4)
zph
ggcoxzph(zph)

### KM of amplified cases stratified by ecDNA+/-, MYCN amp +/-
KM stratified by MYCN amp. log rank tests:
```
             chr+ MYCN+ chr+ MYCN- ecDNA+ MYCN+
chr+ MYCN-   0.12       -          -           
ecDNA+ MYCN+ 0.79       0.05       -           
ecDNA+ MYCN- 0.71       0.24       0.71 
```

In [ ]:
# KM of MYCN_amp_AC x ecDNA
data_subset <- data %>% 
    filter(amp_status == 'amp.') %>%
    mutate(group = case_when(
        ecDNA_status == "ecDNA-" & MYCN_amp_AC == 'nonamp.' ~ "chr+ MYCN-",
        ecDNA_status == "ecDNA-" & MYCN_amp_AC == 'amp.' ~ "chr+ MYCN+",
        ecDNA_status == "ecDNA+" & MYCN_amp_AC == 'nonamp.' ~ "ecDNA+ MYCN-",
        ecDNA_status == "ecDNA+" & MYCN_amp_AC == 'amp.' ~ "ecDNA+ MYCN+"
    ))
formula = Surv(OS_months_5y, OS_status_5y) ~ group
km = survfit2(formula=formula, data = data_subset )
plt <- plotting$km_plot(km)
plt
plotting$save_ggplot("km_nbl_ec_x_mycn")
logrank <- pairwise_survdiff(formula,data_subset,p.adjust.method="BH",rho=0)
logrank
names(km$strata)

### fully parameterized Cox model including MYCN amp
- 80% powered to detect effects of size `exp(se(coef)*2.8)=exp(0.67*2.8)~6.5`, so FN results likely for hazards under 6.5.

In [ ]:
# Cox model including MYCN amp, using rodriguez annotations
m5 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + MYCN_amp + sex + age_at_diagnosis, data = data)
plotting$cox_plot(m5)
zph = cox.zph(m5)
zph
#ggcoxzph(zph)

In [ ]:
# Cox model including MYCN amp, using AmpliconClassifier annotations
m6 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + amp_status + MYCN_amp_AC + sex + age_at_diagnosis, data = data)
#plotting$cox_plot(m6,data,width=6,height=4)
p <- plotting$forest_coxph(m6)
plotting$show_forestploter(p)
plotting$save_forestploter(p,'out/cox_forest_nbl_mycn.png')
plotting$save_forestploter(p,'out/cox_forest_nbl_mycn.svg')
zph = cox.zph(m6)
summary(m6)
zph
ggcoxzph(zph)

In [ ]:
exp(2.8*.67)
1.96*.67

In [ ]:
# VIF: variance inflation from collinearity among the amplification terms in m6.
# GVIF == VIF for these 2-level factors; SE is inflated by sqrt(VIF); effective n ~ n / VIF.
m7 <- coxph(Surv(OS_months_5y, OS_status_5y) ~ ecDNA_status + MYCN_amp_AC + sex + age_at_diagnosis, data = data)
car::vif(m7)